# COMP8325 — Task 2: Holdout Dataset Evaluation

This notebook loads the **already-trained** models saved by `02_model_training_and_comparison.ipynb` and evaluates them on the **holdout dataset** released by the unit convenor.

> **Important:** You are **not** allowed to retrain the models after the holdout dataset is released. Only call `predict()` here — never `fit()`.

**Reported metrics:**
- Overall accuracy
- Per-class TPR (recall) for each malware category (including benign)
- Per-class FPR for each malware category (including benign)

## 1. Imports and configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ── Paths ────────────────────────────────────────────────────────────────────
# Update DATA_DIR to point to the holdout data folder when it is released.
# The holdout .npz is formatted identically to the training dataset.
NPZ_PATH  = "data/bodmas.npz"
META_PATH = "data/bodmas_metadata.csv"
CAT_PATH  = "data/bodmas_malware_category.csv"

ARTIFACTS_DIR = Path("artifacts")

print("Artifact directory:", ARTIFACTS_DIR.resolve())

Artifact directory: C:\Users\vedlo\OneDrive - Macquarie University\COMP8325 Applications of Artificial Intelligence for Cyber Security\Assignement\a2\COMP8325-Assignment2-Group9\artifacts


## 2. Load saved preprocessing objects and trained models

In [2]:
le                = joblib.load(ARTIFACTS_DIR / "label_encoder.joblib")
imputer           = joblib.load(ARTIFACTS_DIR / "median_imputer.joblib")
variance_selector = joblib.load(ARTIFACTS_DIR / "variance_threshold.joblib")
rf_model          = joblib.load(ARTIFACTS_DIR / "random_forest.joblib")
hgb_model         = joblib.load(ARTIFACTS_DIR / "hist_gradient_boosting.joblib")

class_names = le.classes_.tolist()
print(f"Loaded {len(class_names)} classes: {class_names}")

Loaded 15 classes: ['backdoor', 'benign', 'cryptominer', 'downloader', 'dropper', 'exploit', 'informationstealer', 'p2p-worm', 'pua', 'ransomware', 'rootkit', 'trojan', 'trojan-gamethief', 'virus', 'worm']


## 3. Load and preprocess the holdout dataset

Apply the **same** preprocessing pipeline (median imputer + variance threshold) that was **fit on training data** only. Do **not** re-fit anything.

In [3]:
# ── Load raw holdout data ────────────────────────────────────────────────────
raw_h      = np.load(HOLDOUT_NPZ_PATH)
X_holdout  = raw_h["X"]
y_binary_h = raw_h["y"]

meta_h = pd.read_csv(HOLDOUT_META_PATH)
cats_h = pd.read_csv(HOLDOUT_CAT_PATH)

if "sha" in meta_h.columns and "sha256" not in meta_h.columns:
    meta_h = meta_h.rename(columns={"sha": "sha256"})

merged_h   = meta_h.merge(cats_h, on="sha256", how="left")
y_cat_h    = np.where(
    y_binary_h == 0,
    "benign",
    merged_h["category"].astype(str).values,
)

print(f"Holdout X shape : {X_holdout.shape}")
print("Holdout class distribution:")
print(pd.Series(y_cat_h).value_counts().to_string())

NameError: name 'HOLDOUT_NPZ_PATH' is not defined

In [ ]:
# Encode labels using the label encoder fitted during training
y_holdout = le.transform(y_cat_h)

# Apply pre-trained imputer and variance selector (transform only — no fit!)
X_holdout_imp = imputer.transform(X_holdout)
X_holdout_proc = variance_selector.transform(X_holdout_imp)

print(f"Holdout features after preprocessing: {X_holdout_proc.shape[1]}")

## 4. Evaluation helpers (same as training notebook)

In [ ]:
def per_class_tpr_fpr(y_true, y_pred, labels):
    cm  = confusion_matrix(y_true, y_pred, labels=labels)
    tpr, fpr = [], []
    n = cm.sum()
    for k in range(len(labels)):
        tp = cm[k, k]
        fn = cm[k, :].sum() - tp
        fp = cm[:, k].sum() - tp
        tn = n - tp - fn - fp
        tpr.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
        fpr.append(fp / (fp + tn) if (fp + tn) > 0 else 0)
    return np.array(tpr), np.array(fpr)


def metrics_report(name, y_true, y_pred, label_names):
    labels_idx = np.arange(len(label_names))
    acc        = accuracy_score(y_true, y_pred)
    tpr, fpr   = per_class_tpr_fpr(y_true, y_pred, labels_idx)
    print(f"\n=== {name} ===")
    print(f"Overall accuracy: {acc:.4f}")
    df = pd.DataFrame({
        "class":        label_names,
        "TPR (recall)": tpr,
        "FPR":          fpr,
    })
    pd.set_option("display.max_rows", None)
    print(df.to_string(index=False))
    print("\nClassification report:")
    print(classification_report(
        y_true, y_pred, target_names=label_names, digits=4, zero_division=0
    ))
    return acc, df

## 5. Run predictions on holdout (no retraining)

In [ ]:
rf_holdout_pred  = rf_model.predict(X_holdout_proc)
hgb_holdout_pred = hgb_model.predict(X_holdout_proc)

rf_holdout_acc,  rf_holdout_df  = metrics_report(
    "Random Forest (holdout)", y_holdout, rf_holdout_pred, class_names
)
hgb_holdout_acc, hgb_holdout_df = metrics_report(
    "HistGradientBoosting (holdout)", y_holdout, hgb_holdout_pred, class_names
)

## 6. Confusion matrices — holdout set (row-normalised)

In [ ]:
def plot_confusion_row_normalised(
    y_true, y_pred, class_names_list, title, save_path
):
    lab_idx  = np.arange(len(class_names_list))
    cm       = confusion_matrix(y_true, y_pred, labels=lab_idx)
    row_sums = cm.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        cm_norm = np.divide(
            cm, row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0,
        )
    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm_norm, interpolation="nearest", cmap="Blues", vmin=0, vmax=1)
    ax.set_title(title)
    n = len(class_names_list)
    ax.set_xticks(np.arange(n))
    ax.set_yticks(np.arange(n))
    ax.set_xticklabels(class_names_list, rotation=75, ha="right", fontsize=7)
    ax.set_yticklabels(class_names_list, fontsize=7)
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Fraction of true row")
    plt.tight_layout()
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()


plot_confusion_row_normalised(
    y_holdout, rf_holdout_pred, class_names,
    "Random Forest — holdout (rows normalised)",
    "artifacts/cm_rf_holdout.png",
)
plot_confusion_row_normalised(
    y_holdout, hgb_holdout_pred, class_names,
    "HistGradientBoosting — holdout (rows normalised)",
    "artifacts/cm_hgb_holdout.png",
)

## 7. Side-by-side summary and reflection

In [ ]:
summary_h = pd.DataFrame({
    "Model":            ["Random Forest", "HistGradientBoosting"],
    "Holdout accuracy": [rf_holdout_acc, hgb_holdout_acc],
})
print(summary_h.to_string(index=False))

compare_h = pd.DataFrame({
    "class":   class_names,
    "RF_TPR":  rf_holdout_df["TPR (recall)"].values,
    "RF_FPR":  rf_holdout_df["FPR"].values,
    "HGB_TPR": hgb_holdout_df["TPR (recall)"].values,
    "HGB_FPR": hgb_holdout_df["FPR"].values,
})
print("\nPer-class TPR/FPR on holdout:")
print(compare_h.to_string(index=False))

out_dir = Path("artifacts")
summary_h.to_csv(out_dir / "holdout_summary.csv",          index=False)
compare_h.to_csv(out_dir / "per_class_metrics_holdout.csv", index=False)

print("\nSaved holdout artifacts under ./artifacts/")

## 8. Notes for the PDF report (Task 2 section)

Copy the printed holdout metrics into your report and address the following (rubric: 4.5 marks):

1. **Correct use of already-trained classifiers** — confirm no retraining occurred; only `predict()` was called on the holdout set.
2. **Overall accuracy** — state the holdout accuracy for each model and compare it to the test-set accuracy from Task 1.
3. **TPR and FPR per class** — report the table above; highlight any classes where performance drops significantly vs. the test set.
4. **Reflection** — discuss potential reasons for any performance difference between the original test set and the holdout: possible distribution shift, class imbalance in the holdout, rare classes with few samples, etc. Consider whether the gap is within expected variance or signals overfitting.